In [1]:
import os
os.environ['HADOOP_HOME'] = r'C:\hadoop'
os.environ['PATH'] = os.environ['PATH'] + r';C:\hadoop\bin'

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, when, count, avg, max, min, sum,
    upper, lower, length, substring,
    concat_ws, trim, rank, dense_rank,
    row_number, lead, lag
)
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .appName("HospitalAnalytics") \
    .master("local[*]") \
    .config("spark.sql.shuffle.partitions", "2") \
    .getOrCreate()


In [2]:
patients_df = spark.read.csv(
    r"C:\new_env\hexaware-sql\june_11\patients.csv",
    header=True,
    inferSchema=True
)

appointments_df = spark.read.csv(
    r"C:\new_env\hexaware-sql\june_11\appointments.csv",
    header=True,
    inferSchema=True
)


In [4]:
# 1
patients_df.show(truncate=False)


+----------+------------+---------+---+------+-----------+----------------+
|patient_id|patient_name|city     |age|gender|blood_group|insurance_status|
+----------+------------+---------+---+------+-----------+----------------+
|101       |Rahul Sharma|Hyderabad|35 |Male  |O+         |Active          |
|102       |Priya Reddy |Bangalore|29 |Female|A+         |Active          |
|103       |Amit Kumar  |Mumbai   |42 |Male  |B+         |Inactive        |
|104       |Sneha Patel |Chennai  |31 |Female|O+         |Active          |
|105       |Farhan Ali  |Delhi    |55 |Male  |AB+        |Active          |
|106       |Neha Singh  |NULL     |38 |Female|A+         |Inactive        |
|107       |Arjun Verma |Pune     |26 |Male  |B+         |Active          |
|108       |Meera Nair  |Kochi    |48 |Female|O-         |Active          |
|109       |Kiran Rao   |Hyderabad|33 |Male  |NULL       |Inactive        |
|110       |Nisha Reddy |Bangalore|41 |Female|A+         |Active          |
+----------+

In [5]:
# 2
appointments_df.show(truncate=False)


+--------------+----------+-----------+-----------+----------------+----------------+---------+
|appointment_id|patient_id|doctor_name|department |appointment_date|consultation_fee|status   |
+--------------+----------+-----------+-----------+----------------+----------------+---------+
|5001          |101       |Dr. Ramesh |Cardiology |2025-01-10      |1500            |Completed|
|5002          |102       |Dr. Suresh |Neurology  |2025-01-11      |2000            |Completed|
|5003          |101       |Dr. Anita  |Dermatology|2025-01-15      |1000            |Completed|
|5004          |103       |Dr. Ramesh |Cardiology |2025-01-20      |1500            |Cancelled|
|5005          |104       |Dr. Priya  |Orthopedics|2025-01-22      |2500            |Completed|
|5006          |105       |Dr. Anita  |Dermatology|2025-01-25      |1000            |Pending  |
|5007          |107       |Dr. Suresh |Neurology  |2025-02-01      |2000            |Completed|
|5008          |110       |Dr. Priya  |O

In [6]:
# 3
patients_df.printSchema()
appointments_df.printSchema()


root
 |-- patient_id: integer (nullable = true)
 |-- patient_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- blood_group: string (nullable = true)
 |-- insurance_status: string (nullable = true)

root
 |-- appointment_id: integer (nullable = true)
 |-- patient_id: integer (nullable = true)
 |-- doctor_name: string (nullable = true)
 |-- department: string (nullable = true)
 |-- appointment_date: date (nullable = true)
 |-- consultation_fee: integer (nullable = true)
 |-- status: string (nullable = true)



In [7]:
# 4
print('Patients:', patients_df.count())
print('Appointments:', appointments_df.count())


Patients: 10
Appointments: 10


In [8]:
# 5
patients_df.show(5)
appointments_df.show(5)


+----------+------------+---------+---+------+-----------+----------------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|
+----------+------------+---------+---+------+-----------+----------------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|
|       102| Priya Reddy|Bangalore| 29|Female|         A+|          Active|
|       103|  Amit Kumar|   Mumbai| 42|  Male|         B+|        Inactive|
|       104| Sneha Patel|  Chennai| 31|Female|         O+|          Active|
|       105|  Farhan Ali|    Delhi| 55|  Male|        AB+|          Active|
+----------+------------+---------+---+------+-----------+----------------+
only showing top 5 rows
+--------------+----------+-----------+-----------+----------------+----------------+---------+
|appointment_id|patient_id|doctor_name| department|appointment_date|consultation_fee|   status|
+--------------+----------+-----------+-----------+----------------+----------------+---------+
|   

In [9]:
# 6
patients_df.select('city').distinct().show()


+---------+
|     city|
+---------+
|Hyderabad|
|Bangalore|
|     Pune|
|   Mumbai|
|  Chennai|
|    Delhi|
|    Kochi|
|     NULL|
+---------+



In [10]:
# 7
appointments_df.select('department').distinct().show()


+-----------+
| department|
+-----------+
| Cardiology|
|  Neurology|
|Dermatology|
|Orthopedics|
+-----------+



In [11]:
# 8
patients_df.write.mode('overwrite').parquet(r'C:\new_env\hexaware-sql\june_11\patients.parquet')
print('Saved')


Saved


In [12]:
# 9
parquet_df = spark.read.parquet(r'C:\new_env\hexaware-sql\june_11\patients.parquet')
parquet_df.show(truncate=False)


+----------+------------+---------+---+------+-----------+----------------+
|patient_id|patient_name|city     |age|gender|blood_group|insurance_status|
+----------+------------+---------+---+------+-----------+----------------+
|101       |Rahul Sharma|Hyderabad|35 |Male  |O+         |Active          |
|102       |Priya Reddy |Bangalore|29 |Female|A+         |Active          |
|103       |Amit Kumar  |Mumbai   |42 |Male  |B+         |Inactive        |
|104       |Sneha Patel |Chennai  |31 |Female|O+         |Active          |
|105       |Farhan Ali  |Delhi    |55 |Male  |AB+        |Active          |
|106       |Neha Singh  |NULL     |38 |Female|A+         |Inactive        |
|107       |Arjun Verma |Pune     |26 |Male  |B+         |Active          |
|108       |Meera Nair  |Kochi    |48 |Female|O-         |Active          |
|109       |Kiran Rao   |Hyderabad|33 |Male  |NULL       |Inactive        |
|110       |Nisha Reddy |Bangalore|41 |Female|A+         |Active          |
+----------+

In [13]:
# 10
print('CSV count:', patients_df.count())
print('Parquet count:', spark.read.parquet(r'C:\new_env\hexaware-sql\june_11\patients.parquet').count())


CSV count: 10
Parquet count: 10


In [14]:
# 11
patients_df.filter(col('city') == 'Hyderabad').show(truncate=False)


+----------+------------+---------+---+------+-----------+----------------+
|patient_id|patient_name|city     |age|gender|blood_group|insurance_status|
+----------+------------+---------+---+------+-----------+----------------+
|101       |Rahul Sharma|Hyderabad|35 |Male  |O+         |Active          |
|109       |Kiran Rao   |Hyderabad|33 |Male  |NULL       |Inactive        |
+----------+------------+---------+---+------+-----------+----------------+



In [15]:
# 12
patients_df.filter(col('gender') == 'Female').show(truncate=False)


+----------+------------+---------+---+------+-----------+----------------+
|patient_id|patient_name|city     |age|gender|blood_group|insurance_status|
+----------+------------+---------+---+------+-----------+----------------+
|102       |Priya Reddy |Bangalore|29 |Female|A+         |Active          |
|104       |Sneha Patel |Chennai  |31 |Female|O+         |Active          |
|106       |Neha Singh  |NULL     |38 |Female|A+         |Inactive        |
|108       |Meera Nair  |Kochi    |48 |Female|O-         |Active          |
|110       |Nisha Reddy |Bangalore|41 |Female|A+         |Active          |
+----------+------------+---------+---+------+-----------+----------------+



In [16]:
# 13
patients_df.filter(col('age') > 40).show(truncate=False)


+----------+------------+---------+---+------+-----------+----------------+
|patient_id|patient_name|city     |age|gender|blood_group|insurance_status|
+----------+------------+---------+---+------+-----------+----------------+
|103       |Amit Kumar  |Mumbai   |42 |Male  |B+         |Inactive        |
|105       |Farhan Ali  |Delhi    |55 |Male  |AB+        |Active          |
|108       |Meera Nair  |Kochi    |48 |Female|O-         |Active          |
|110       |Nisha Reddy |Bangalore|41 |Female|A+         |Active          |
+----------+------------+---------+---+------+-----------+----------------+



In [17]:
# 14
appointments_df.filter(col('status') == 'Completed').show(truncate=False)


+--------------+----------+-----------+-----------+----------------+----------------+---------+
|appointment_id|patient_id|doctor_name|department |appointment_date|consultation_fee|status   |
+--------------+----------+-----------+-----------+----------------+----------------+---------+
|5001          |101       |Dr. Ramesh |Cardiology |2025-01-10      |1500            |Completed|
|5002          |102       |Dr. Suresh |Neurology  |2025-01-11      |2000            |Completed|
|5003          |101       |Dr. Anita  |Dermatology|2025-01-15      |1000            |Completed|
|5005          |104       |Dr. Priya  |Orthopedics|2025-01-22      |2500            |Completed|
|5007          |107       |Dr. Suresh |Neurology  |2025-02-01      |2000            |Completed|
|5008          |110       |Dr. Priya  |Orthopedics|2025-02-03      |2500            |Completed|
|5009          |120       |Dr. Ramesh |Cardiology |2025-02-05      |1500            |Completed|
+--------------+----------+-----------+-

In [18]:
# 15
appointments_df.filter(col('status') == 'Pending').show(truncate=False)


+--------------+----------+-----------+-----------+----------------+----------------+-------+
|appointment_id|patient_id|doctor_name|department |appointment_date|consultation_fee|status |
+--------------+----------+-----------+-----------+----------------+----------------+-------+
|5006          |105       |Dr. Anita  |Dermatology|2025-01-25      |1000            |Pending|
|5010          |108       |Dr. Anita  |Dermatology|2025-02-10      |NULL            |Pending|
+--------------+----------+-----------+-----------+----------------+----------------+-------+



In [19]:
# 16
appointments_df.filter(col('consultation_fee') > 1500).show(truncate=False)


+--------------+----------+-----------+-----------+----------------+----------------+---------+
|appointment_id|patient_id|doctor_name|department |appointment_date|consultation_fee|status   |
+--------------+----------+-----------+-----------+----------------+----------------+---------+
|5002          |102       |Dr. Suresh |Neurology  |2025-01-11      |2000            |Completed|
|5005          |104       |Dr. Priya  |Orthopedics|2025-01-22      |2500            |Completed|
|5007          |107       |Dr. Suresh |Neurology  |2025-02-01      |2000            |Completed|
|5008          |110       |Dr. Priya  |Orthopedics|2025-02-03      |2500            |Completed|
+--------------+----------+-----------+-----------+----------------+----------------+---------+



In [20]:
# 17
patients_df.filter(col('insurance_status') == 'Active').show(truncate=False)


+----------+------------+---------+---+------+-----------+----------------+
|patient_id|patient_name|city     |age|gender|blood_group|insurance_status|
+----------+------------+---------+---+------+-----------+----------------+
|101       |Rahul Sharma|Hyderabad|35 |Male  |O+         |Active          |
|102       |Priya Reddy |Bangalore|29 |Female|A+         |Active          |
|104       |Sneha Patel |Chennai  |31 |Female|O+         |Active          |
|105       |Farhan Ali  |Delhi    |55 |Male  |AB+        |Active          |
|107       |Arjun Verma |Pune     |26 |Male  |B+         |Active          |
|108       |Meera Nair  |Kochi    |48 |Female|O-         |Active          |
|110       |Nisha Reddy |Bangalore|41 |Female|A+         |Active          |
+----------+------------+---------+---+------+-----------+----------------+



In [21]:
# 18
patients_df.filter(col('insurance_status') == 'Inactive').show(truncate=False)


+----------+------------+---------+---+------+-----------+----------------+
|patient_id|patient_name|city     |age|gender|blood_group|insurance_status|
+----------+------------+---------+---+------+-----------+----------------+
|103       |Amit Kumar  |Mumbai   |42 |Male  |B+         |Inactive        |
|106       |Neha Singh  |NULL     |38 |Female|A+         |Inactive        |
|109       |Kiran Rao   |Hyderabad|33 |Male  |NULL       |Inactive        |
+----------+------------+---------+---+------+-----------+----------------+



In [22]:
# 19
patients_df.filter(col('blood_group') == 'O+').show(truncate=False)


+----------+------------+---------+---+------+-----------+----------------+
|patient_id|patient_name|city     |age|gender|blood_group|insurance_status|
+----------+------------+---------+---+------+-----------+----------------+
|101       |Rahul Sharma|Hyderabad|35 |Male  |O+         |Active          |
|104       |Sneha Patel |Chennai  |31 |Female|O+         |Active          |
+----------+------------+---------+---+------+-----------+----------------+



In [23]:
# 20
appointments_df.filter(col('department') == 'Cardiology').show(truncate=False)


+--------------+----------+-----------+----------+----------------+----------------+---------+
|appointment_id|patient_id|doctor_name|department|appointment_date|consultation_fee|status   |
+--------------+----------+-----------+----------+----------------+----------------+---------+
|5001          |101       |Dr. Ramesh |Cardiology|2025-01-10      |1500            |Completed|
|5004          |103       |Dr. Ramesh |Cardiology|2025-01-20      |1500            |Cancelled|
|5009          |120       |Dr. Ramesh |Cardiology|2025-02-05      |1500            |Completed|
+--------------+----------+-----------+----------+----------------+----------------+---------+



In [24]:
# 21
patients_df.filter(col('city').isNull()).show(truncate=False)


+----------+------------+----+---+------+-----------+----------------+
|patient_id|patient_name|city|age|gender|blood_group|insurance_status|
+----------+------------+----+---+------+-----------+----------------+
|106       |Neha Singh  |NULL|38 |Female|A+         |Inactive        |
+----------+------------+----+---+------+-----------+----------------+



In [25]:
# 22
patients_df.filter(col('blood_group').isNull()).show(truncate=False)


+----------+------------+---------+---+------+-----------+----------------+
|patient_id|patient_name|city     |age|gender|blood_group|insurance_status|
+----------+------------+---------+---+------+-----------+----------------+
|109       |Kiran Rao   |Hyderabad|33 |Male  |NULL       |Inactive        |
+----------+------------+---------+---+------+-----------+----------------+



In [26]:
# 23
appointments_df.filter(col('consultation_fee').isNull()).show(truncate=False)


+--------------+----------+-----------+-----------+----------------+----------------+-------+
|appointment_id|patient_id|doctor_name|department |appointment_date|consultation_fee|status |
+--------------+----------+-----------+-----------+----------------+----------------+-------+
|5010          |108       |Dr. Anita  |Dermatology|2025-02-10      |NULL            |Pending|
+--------------+----------+-----------+-----------+----------------+----------------+-------+



In [27]:
# 24
patients_df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in patients_df.columns
]).show()

appointments_df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in appointments_df.columns
]).show()


+----------+------------+----+---+------+-----------+----------------+
|patient_id|patient_name|city|age|gender|blood_group|insurance_status|
+----------+------------+----+---+------+-----------+----------------+
|         0|           0|   1|  0|     0|          1|               0|
+----------+------------+----+---+------+-----------+----------------+

+--------------+----------+-----------+----------+----------------+----------------+------+
|appointment_id|patient_id|doctor_name|department|appointment_date|consultation_fee|status|
+--------------+----------+-----------+----------+----------------+----------------+------+
|             0|         0|          0|         0|               0|               1|     0|
+--------------+----------+-----------+----------+----------------+----------------+------+



In [28]:
# 25
patients_df.fillna({'city': 'Unknown'}).show(truncate=False)


+----------+------------+---------+---+------+-----------+----------------+
|patient_id|patient_name|city     |age|gender|blood_group|insurance_status|
+----------+------------+---------+---+------+-----------+----------------+
|101       |Rahul Sharma|Hyderabad|35 |Male  |O+         |Active          |
|102       |Priya Reddy |Bangalore|29 |Female|A+         |Active          |
|103       |Amit Kumar  |Mumbai   |42 |Male  |B+         |Inactive        |
|104       |Sneha Patel |Chennai  |31 |Female|O+         |Active          |
|105       |Farhan Ali  |Delhi    |55 |Male  |AB+        |Active          |
|106       |Neha Singh  |Unknown  |38 |Female|A+         |Inactive        |
|107       |Arjun Verma |Pune     |26 |Male  |B+         |Active          |
|108       |Meera Nair  |Kochi    |48 |Female|O-         |Active          |
|109       |Kiran Rao   |Hyderabad|33 |Male  |NULL       |Inactive        |
|110       |Nisha Reddy |Bangalore|41 |Female|A+         |Active          |
+----------+

In [29]:
# 26
patients_df.fillna({'blood_group': 'Not Available'}).show(truncate=False)


+----------+------------+---------+---+------+-------------+----------------+
|patient_id|patient_name|city     |age|gender|blood_group  |insurance_status|
+----------+------------+---------+---+------+-------------+----------------+
|101       |Rahul Sharma|Hyderabad|35 |Male  |O+           |Active          |
|102       |Priya Reddy |Bangalore|29 |Female|A+           |Active          |
|103       |Amit Kumar  |Mumbai   |42 |Male  |B+           |Inactive        |
|104       |Sneha Patel |Chennai  |31 |Female|O+           |Active          |
|105       |Farhan Ali  |Delhi    |55 |Male  |AB+          |Active          |
|106       |Neha Singh  |NULL     |38 |Female|A+           |Inactive        |
|107       |Arjun Verma |Pune     |26 |Male  |B+           |Active          |
|108       |Meera Nair  |Kochi    |48 |Female|O-           |Active          |
|109       |Kiran Rao   |Hyderabad|33 |Male  |Not Available|Inactive        |
|110       |Nisha Reddy |Bangalore|41 |Female|A+           |Acti

In [30]:
# 27
appointments_df.fillna({'consultation_fee': 0}).show(truncate=False)


+--------------+----------+-----------+-----------+----------------+----------------+---------+
|appointment_id|patient_id|doctor_name|department |appointment_date|consultation_fee|status   |
+--------------+----------+-----------+-----------+----------------+----------------+---------+
|5001          |101       |Dr. Ramesh |Cardiology |2025-01-10      |1500            |Completed|
|5002          |102       |Dr. Suresh |Neurology  |2025-01-11      |2000            |Completed|
|5003          |101       |Dr. Anita  |Dermatology|2025-01-15      |1000            |Completed|
|5004          |103       |Dr. Ramesh |Cardiology |2025-01-20      |1500            |Cancelled|
|5005          |104       |Dr. Priya  |Orthopedics|2025-01-22      |2500            |Completed|
|5006          |105       |Dr. Anita  |Dermatology|2025-01-25      |1000            |Pending  |
|5007          |107       |Dr. Suresh |Neurology  |2025-02-01      |2000            |Completed|
|5008          |110       |Dr. Priya  |O

In [31]:
# 28
appointments_df.dropna(subset=['consultation_fee']).show(truncate=False)


+--------------+----------+-----------+-----------+----------------+----------------+---------+
|appointment_id|patient_id|doctor_name|department |appointment_date|consultation_fee|status   |
+--------------+----------+-----------+-----------+----------------+----------------+---------+
|5001          |101       |Dr. Ramesh |Cardiology |2025-01-10      |1500            |Completed|
|5002          |102       |Dr. Suresh |Neurology  |2025-01-11      |2000            |Completed|
|5003          |101       |Dr. Anita  |Dermatology|2025-01-15      |1000            |Completed|
|5004          |103       |Dr. Ramesh |Cardiology |2025-01-20      |1500            |Cancelled|
|5005          |104       |Dr. Priya  |Orthopedics|2025-01-22      |2500            |Completed|
|5006          |105       |Dr. Anita  |Dermatology|2025-01-25      |1000            |Pending  |
|5007          |107       |Dr. Suresh |Neurology  |2025-02-01      |2000            |Completed|
|5008          |110       |Dr. Priya  |O

In [32]:
# 29
patients_df.withColumn(
    'data_quality_status',
    when(
        col('city').isNull() | col('blood_group').isNull(),
        'Incomplete'
    ).otherwise('Complete')
).show(truncate=False)


+----------+------------+---------+---+------+-----------+----------------+-------------------+
|patient_id|patient_name|city     |age|gender|blood_group|insurance_status|data_quality_status|
+----------+------------+---------+---+------+-----------+----------------+-------------------+
|101       |Rahul Sharma|Hyderabad|35 |Male  |O+         |Active          |Complete           |
|102       |Priya Reddy |Bangalore|29 |Female|A+         |Active          |Complete           |
|103       |Amit Kumar  |Mumbai   |42 |Male  |B+         |Inactive        |Complete           |
|104       |Sneha Patel |Chennai  |31 |Female|O+         |Active          |Complete           |
|105       |Farhan Ali  |Delhi    |55 |Male  |AB+        |Active          |Complete           |
|106       |Neha Singh  |NULL     |38 |Female|A+         |Inactive        |Incomplete         |
|107       |Arjun Verma |Pune     |26 |Male  |B+         |Active          |Complete           |
|108       |Meera Nair  |Kochi    |48 |F

In [33]:
# 30
patients_df.withColumn(
    'data_quality_status',
    when(
        col('city').isNull() | col('blood_group').isNull(),
        'Incomplete'
    ).otherwise('Complete')
).groupBy('data_quality_status').count().show()


+-------------------+-----+
|data_quality_status|count|
+-------------------+-----+
|           Complete|    8|
|         Incomplete|    2|
+-------------------+-----+



In [34]:
# 31
patients_df.withColumn('patient_name', upper(col('patient_name'))).show(truncate=False)


+----------+------------+---------+---+------+-----------+----------------+
|patient_id|patient_name|city     |age|gender|blood_group|insurance_status|
+----------+------------+---------+---+------+-----------+----------------+
|101       |RAHUL SHARMA|Hyderabad|35 |Male  |O+         |Active          |
|102       |PRIYA REDDY |Bangalore|29 |Female|A+         |Active          |
|103       |AMIT KUMAR  |Mumbai   |42 |Male  |B+         |Inactive        |
|104       |SNEHA PATEL |Chennai  |31 |Female|O+         |Active          |
|105       |FARHAN ALI  |Delhi    |55 |Male  |AB+        |Active          |
|106       |NEHA SINGH  |NULL     |38 |Female|A+         |Inactive        |
|107       |ARJUN VERMA |Pune     |26 |Male  |B+         |Active          |
|108       |MEERA NAIR  |Kochi    |48 |Female|O-         |Active          |
|109       |KIRAN RAO   |Hyderabad|33 |Male  |NULL       |Inactive        |
|110       |NISHA REDDY |Bangalore|41 |Female|A+         |Active          |
+----------+

In [35]:
# 32
patients_df.withColumn('patient_name', lower(col('patient_name'))).show(truncate=False)


+----------+------------+---------+---+------+-----------+----------------+
|patient_id|patient_name|city     |age|gender|blood_group|insurance_status|
+----------+------------+---------+---+------+-----------+----------------+
|101       |rahul sharma|Hyderabad|35 |Male  |O+         |Active          |
|102       |priya reddy |Bangalore|29 |Female|A+         |Active          |
|103       |amit kumar  |Mumbai   |42 |Male  |B+         |Inactive        |
|104       |sneha patel |Chennai  |31 |Female|O+         |Active          |
|105       |farhan ali  |Delhi    |55 |Male  |AB+        |Active          |
|106       |neha singh  |NULL     |38 |Female|A+         |Inactive        |
|107       |arjun verma |Pune     |26 |Male  |B+         |Active          |
|108       |meera nair  |Kochi    |48 |Female|O-         |Active          |
|109       |kiran rao   |Hyderabad|33 |Male  |NULL       |Inactive        |
|110       |nisha reddy |Bangalore|41 |Female|A+         |Active          |
+----------+

In [36]:
# 33
patients_df.withColumn('name_length', length(col('patient_name'))).show(truncate=False)


+----------+------------+---------+---+------+-----------+----------------+-----------+
|patient_id|patient_name|city     |age|gender|blood_group|insurance_status|name_length|
+----------+------------+---------+---+------+-----------+----------------+-----------+
|101       |Rahul Sharma|Hyderabad|35 |Male  |O+         |Active          |12         |
|102       |Priya Reddy |Bangalore|29 |Female|A+         |Active          |11         |
|103       |Amit Kumar  |Mumbai   |42 |Male  |B+         |Inactive        |10         |
|104       |Sneha Patel |Chennai  |31 |Female|O+         |Active          |11         |
|105       |Farhan Ali  |Delhi    |55 |Male  |AB+        |Active          |10         |
|106       |Neha Singh  |NULL     |38 |Female|A+         |Inactive        |10         |
|107       |Arjun Verma |Pune     |26 |Male  |B+         |Active          |11         |
|108       |Meera Nair  |Kochi    |48 |Female|O-         |Active          |10         |
|109       |Kiran Rao   |Hyderab

In [37]:
# 34
patients_df.withColumn('first_3', substring(col('patient_name'), 1, 3)).show(truncate=False)


+----------+------------+---------+---+------+-----------+----------------+-------+
|patient_id|patient_name|city     |age|gender|blood_group|insurance_status|first_3|
+----------+------------+---------+---+------+-----------+----------------+-------+
|101       |Rahul Sharma|Hyderabad|35 |Male  |O+         |Active          |Rah    |
|102       |Priya Reddy |Bangalore|29 |Female|A+         |Active          |Pri    |
|103       |Amit Kumar  |Mumbai   |42 |Male  |B+         |Inactive        |Ami    |
|104       |Sneha Patel |Chennai  |31 |Female|O+         |Active          |Sne    |
|105       |Farhan Ali  |Delhi    |55 |Male  |AB+        |Active          |Far    |
|106       |Neha Singh  |NULL     |38 |Female|A+         |Inactive        |Neh    |
|107       |Arjun Verma |Pune     |26 |Male  |B+         |Active          |Arj    |
|108       |Meera Nair  |Kochi    |48 |Female|O-         |Active          |Mee    |
|109       |Kiran Rao   |Hyderabad|33 |Male  |NULL       |Inactive        |K

In [38]:
# 35
patients_df.withColumn(
    'age_group',
    when(col('age') < 18, 'Minor')
    .when(col('age') < 40, 'Adult')
    .when(col('age') < 60, 'Middle-Aged')
    .otherwise('Senior')
).show(truncate=False)


+----------+------------+---------+---+------+-----------+----------------+-----------+
|patient_id|patient_name|city     |age|gender|blood_group|insurance_status|age_group  |
+----------+------------+---------+---+------+-----------+----------------+-----------+
|101       |Rahul Sharma|Hyderabad|35 |Male  |O+         |Active          |Adult      |
|102       |Priya Reddy |Bangalore|29 |Female|A+         |Active          |Adult      |
|103       |Amit Kumar  |Mumbai   |42 |Male  |B+         |Inactive        |Middle-Aged|
|104       |Sneha Patel |Chennai  |31 |Female|O+         |Active          |Adult      |
|105       |Farhan Ali  |Delhi    |55 |Male  |AB+        |Active          |Middle-Aged|
|106       |Neha Singh  |NULL     |38 |Female|A+         |Inactive        |Adult      |
|107       |Arjun Verma |Pune     |26 |Male  |B+         |Active          |Adult      |
|108       |Meera Nair  |Kochi    |48 |Female|O-         |Active          |Middle-Aged|
|109       |Kiran Rao   |Hyderab

In [39]:
# 36
patients_df.withColumn(
    'insurance_flag',
    when(col('insurance_status') == 'Active', 1).otherwise(0)
).show(truncate=False)


+----------+------------+---------+---+------+-----------+----------------+--------------+
|patient_id|patient_name|city     |age|gender|blood_group|insurance_status|insurance_flag|
+----------+------------+---------+---+------+-----------+----------------+--------------+
|101       |Rahul Sharma|Hyderabad|35 |Male  |O+         |Active          |1             |
|102       |Priya Reddy |Bangalore|29 |Female|A+         |Active          |1             |
|103       |Amit Kumar  |Mumbai   |42 |Male  |B+         |Inactive        |0             |
|104       |Sneha Patel |Chennai  |31 |Female|O+         |Active          |1             |
|105       |Farhan Ali  |Delhi    |55 |Male  |AB+        |Active          |1             |
|106       |Neha Singh  |NULL     |38 |Female|A+         |Inactive        |0             |
|107       |Arjun Verma |Pune     |26 |Male  |B+         |Active          |1             |
|108       |Meera Nair  |Kochi    |48 |Female|O-         |Active          |1             |

In [40]:
# 37
patients_df.withColumn(
    'senior_citizen',
    when(col('age') >= 60, 'Yes').otherwise('No')
).show(truncate=False)


+----------+------------+---------+---+------+-----------+----------------+--------------+
|patient_id|patient_name|city     |age|gender|blood_group|insurance_status|senior_citizen|
+----------+------------+---------+---+------+-----------+----------------+--------------+
|101       |Rahul Sharma|Hyderabad|35 |Male  |O+         |Active          |No            |
|102       |Priya Reddy |Bangalore|29 |Female|A+         |Active          |No            |
|103       |Amit Kumar  |Mumbai   |42 |Male  |B+         |Inactive        |No            |
|104       |Sneha Patel |Chennai  |31 |Female|O+         |Active          |No            |
|105       |Farhan Ali  |Delhi    |55 |Male  |AB+        |Active          |No            |
|106       |Neha Singh  |NULL     |38 |Female|A+         |Inactive        |No            |
|107       |Arjun Verma |Pune     |26 |Male  |B+         |Active          |No            |
|108       |Meera Nair  |Kochi    |48 |Female|O-         |Active          |No            |

In [41]:
# 38
patients_df.withColumn(
    'name_city',
    concat_ws(' - ', col('patient_name'), col('city'))
).show(truncate=False)


+----------+------------+---------+---+------+-----------+----------------+------------------------+
|patient_id|patient_name|city     |age|gender|blood_group|insurance_status|name_city               |
+----------+------------+---------+---+------+-----------+----------------+------------------------+
|101       |Rahul Sharma|Hyderabad|35 |Male  |O+         |Active          |Rahul Sharma - Hyderabad|
|102       |Priya Reddy |Bangalore|29 |Female|A+         |Active          |Priya Reddy - Bangalore |
|103       |Amit Kumar  |Mumbai   |42 |Male  |B+         |Inactive        |Amit Kumar - Mumbai     |
|104       |Sneha Patel |Chennai  |31 |Female|O+         |Active          |Sneha Patel - Chennai   |
|105       |Farhan Ali  |Delhi    |55 |Male  |AB+        |Active          |Farhan Ali - Delhi      |
|106       |Neha Singh  |NULL     |38 |Female|A+         |Inactive        |Neha Singh              |
|107       |Arjun Verma |Pune     |26 |Male  |B+         |Active          |Arjun Verma - Pu

In [42]:
# 39
patients_df.withColumn('patient_name', trim(col('patient_name'))).show(truncate=False)


+----------+------------+---------+---+------+-----------+----------------+
|patient_id|patient_name|city     |age|gender|blood_group|insurance_status|
+----------+------------+---------+---+------+-----------+----------------+
|101       |Rahul Sharma|Hyderabad|35 |Male  |O+         |Active          |
|102       |Priya Reddy |Bangalore|29 |Female|A+         |Active          |
|103       |Amit Kumar  |Mumbai   |42 |Male  |B+         |Inactive        |
|104       |Sneha Patel |Chennai  |31 |Female|O+         |Active          |
|105       |Farhan Ali  |Delhi    |55 |Male  |AB+        |Active          |
|106       |Neha Singh  |NULL     |38 |Female|A+         |Inactive        |
|107       |Arjun Verma |Pune     |26 |Male  |B+         |Active          |
|108       |Meera Nair  |Kochi    |48 |Female|O-         |Active          |
|109       |Kiran Rao   |Hyderabad|33 |Male  |NULL       |Inactive        |
|110       |Nisha Reddy |Bangalore|41 |Female|A+         |Active          |
+----------+

In [43]:
# 40
patients_df.withColumn('city', upper(col('city'))).show(truncate=False)


+----------+------------+---------+---+------+-----------+----------------+
|patient_id|patient_name|city     |age|gender|blood_group|insurance_status|
+----------+------------+---------+---+------+-----------+----------------+
|101       |Rahul Sharma|HYDERABAD|35 |Male  |O+         |Active          |
|102       |Priya Reddy |BANGALORE|29 |Female|A+         |Active          |
|103       |Amit Kumar  |MUMBAI   |42 |Male  |B+         |Inactive        |
|104       |Sneha Patel |CHENNAI  |31 |Female|O+         |Active          |
|105       |Farhan Ali  |DELHI    |55 |Male  |AB+        |Active          |
|106       |Neha Singh  |NULL     |38 |Female|A+         |Inactive        |
|107       |Arjun Verma |PUNE     |26 |Male  |B+         |Active          |
|108       |Meera Nair  |KOCHI    |48 |Female|O-         |Active          |
|109       |Kiran Rao   |HYDERABAD|33 |Male  |NULL       |Inactive        |
|110       |Nisha Reddy |BANGALORE|41 |Female|A+         |Active          |
+----------+

In [44]:
# 41
patients_df.groupBy('city').count().show()


+---------+-----+
|     city|count|
+---------+-----+
|Hyderabad|    2|
|Bangalore|    2|
|     Pune|    1|
|     NULL|    1|
|   Mumbai|    1|
|  Chennai|    1|
|    Delhi|    1|
|    Kochi|    1|
+---------+-----+



In [45]:
# 42
patients_df.groupBy('gender').count().show()


+------+-----+
|gender|count|
+------+-----+
|Female|    5|
|  Male|    5|
+------+-----+



In [46]:
# 43
patients_df.groupBy('blood_group').count().show()


+-----------+-----+
|blood_group|count|
+-----------+-----+
|         B+|    2|
|         O-|    1|
|       NULL|    1|
|         O+|    2|
|         A+|    3|
|        AB+|    1|
+-----------+-----+



In [47]:
# 44
appointments_df.groupBy('department').count().show()


+-----------+-----+
| department|count|
+-----------+-----+
| Cardiology|    3|
|  Neurology|    2|
|Dermatology|    3|
|Orthopedics|    2|
+-----------+-----+



In [48]:
# 45
patients_df.groupBy('city').avg('age').show()


+---------+--------+
|     city|avg(age)|
+---------+--------+
|Hyderabad|    34.0|
|Bangalore|    35.0|
|     Pune|    26.0|
|     NULL|    38.0|
|   Mumbai|    42.0|
|  Chennai|    31.0|
|    Delhi|    55.0|
|    Kochi|    48.0|
+---------+--------+



In [49]:
# 46
patients_df.groupBy('city').max('age').show()


+---------+--------+
|     city|max(age)|
+---------+--------+
|Hyderabad|      35|
|Bangalore|      41|
|     Pune|      26|
|     NULL|      38|
|   Mumbai|      42|
|  Chennai|      31|
|    Delhi|      55|
|    Kochi|      48|
+---------+--------+



In [50]:
# 47
patients_df.groupBy('city').min('age').show()


+---------+--------+
|     city|min(age)|
+---------+--------+
|Hyderabad|      33|
|Bangalore|      29|
|     Pune|      26|
|     NULL|      38|
|   Mumbai|      42|
|  Chennai|      31|
|    Delhi|      55|
|    Kochi|      48|
+---------+--------+



In [51]:
# 48
appointments_df.groupBy('department').avg('consultation_fee').show()


+-----------+---------------------+
| department|avg(consultation_fee)|
+-----------+---------------------+
| Cardiology|               1500.0|
|  Neurology|               2000.0|
|Dermatology|               1000.0|
|Orthopedics|               2500.0|
+-----------+---------------------+



In [52]:
# 49
appointments_df.groupBy('department').sum('consultation_fee').show()


+-----------+---------------------+
| department|sum(consultation_fee)|
+-----------+---------------------+
| Cardiology|                 4500|
|  Neurology|                 4000|
|Dermatology|                 2000|
|Orthopedics|                 5000|
+-----------+---------------------+



In [53]:
# 50
appointments_df.groupBy('department').sum('consultation_fee').orderBy(col('sum(consultation_fee)').desc()).show(1)


+-----------+---------------------+
| department|sum(consultation_fee)|
+-----------+---------------------+
|Orthopedics|                 5000|
+-----------+---------------------+
only showing top 1 row


In [54]:
# 51
inner_df = patients_df.join(
    appointments_df,
    on='patient_id',
    how='inner'
)
inner_df.show(truncate=False)


+----------+------------+---------+---+------+-----------+----------------+--------------+-----------+-----------+----------------+----------------+---------+
|patient_id|patient_name|city     |age|gender|blood_group|insurance_status|appointment_id|doctor_name|department |appointment_date|consultation_fee|status   |
+----------+------------+---------+---+------+-----------+----------------+--------------+-----------+-----------+----------------+----------------+---------+
|101       |Rahul Sharma|Hyderabad|35 |Male  |O+         |Active          |5001          |Dr. Ramesh |Cardiology |2025-01-10      |1500            |Completed|
|102       |Priya Reddy |Bangalore|29 |Female|A+         |Active          |5002          |Dr. Suresh |Neurology  |2025-01-11      |2000            |Completed|
|101       |Rahul Sharma|Hyderabad|35 |Male  |O+         |Active          |5003          |Dr. Anita  |Dermatology|2025-01-15      |1000            |Completed|
|103       |Amit Kumar  |Mumbai   |42 |Male  |

In [55]:
# 52
left_df = patients_df.join(
    appointments_df,
    on='patient_id',
    how='left'
)
left_df.show(truncate=False)


+----------+------------+---------+---+------+-----------+----------------+--------------+-----------+-----------+----------------+----------------+---------+
|patient_id|patient_name|city     |age|gender|blood_group|insurance_status|appointment_id|doctor_name|department |appointment_date|consultation_fee|status   |
+----------+------------+---------+---+------+-----------+----------------+--------------+-----------+-----------+----------------+----------------+---------+
|101       |Rahul Sharma|Hyderabad|35 |Male  |O+         |Active          |5003          |Dr. Anita  |Dermatology|2025-01-15      |1000            |Completed|
|101       |Rahul Sharma|Hyderabad|35 |Male  |O+         |Active          |5001          |Dr. Ramesh |Cardiology |2025-01-10      |1500            |Completed|
|102       |Priya Reddy |Bangalore|29 |Female|A+         |Active          |5002          |Dr. Suresh |Neurology  |2025-01-11      |2000            |Completed|
|103       |Amit Kumar  |Mumbai   |42 |Male  |

In [56]:
# 53
right_df = patients_df.join(
    appointments_df,
    on='patient_id',
    how='right'
)
right_df.show(truncate=False)


+----------+------------+---------+----+------+-----------+----------------+--------------+-----------+-----------+----------------+----------------+---------+
|patient_id|patient_name|city     |age |gender|blood_group|insurance_status|appointment_id|doctor_name|department |appointment_date|consultation_fee|status   |
+----------+------------+---------+----+------+-----------+----------------+--------------+-----------+-----------+----------------+----------------+---------+
|101       |Rahul Sharma|Hyderabad|35  |Male  |O+         |Active          |5001          |Dr. Ramesh |Cardiology |2025-01-10      |1500            |Completed|
|102       |Priya Reddy |Bangalore|29  |Female|A+         |Active          |5002          |Dr. Suresh |Neurology  |2025-01-11      |2000            |Completed|
|101       |Rahul Sharma|Hyderabad|35  |Male  |O+         |Active          |5003          |Dr. Anita  |Dermatology|2025-01-15      |1000            |Completed|
|103       |Amit Kumar  |Mumbai   |42  |

In [57]:
# 54
full_df = patients_df.join(
    appointments_df,
    on='patient_id',
    how='full'
)
full_df.show(truncate=False)


+----------+------------+---------+----+------+-----------+----------------+--------------+-----------+-----------+----------------+----------------+---------+
|patient_id|patient_name|city     |age |gender|blood_group|insurance_status|appointment_id|doctor_name|department |appointment_date|consultation_fee|status   |
+----------+------------+---------+----+------+-----------+----------------+--------------+-----------+-----------+----------------+----------------+---------+
|101       |Rahul Sharma|Hyderabad|35  |Male  |O+         |Active          |5001          |Dr. Ramesh |Cardiology |2025-01-10      |1500            |Completed|
|101       |Rahul Sharma|Hyderabad|35  |Male  |O+         |Active          |5003          |Dr. Anita  |Dermatology|2025-01-15      |1000            |Completed|
|102       |Priya Reddy |Bangalore|29  |Female|A+         |Active          |5002          |Dr. Suresh |Neurology  |2025-01-11      |2000            |Completed|
|103       |Amit Kumar  |Mumbai   |42  |

In [58]:
# 55
no_appt = patients_df.join(
    appointments_df,
    on='patient_id',
    how='left'
).filter(col('appointment_id').isNull())
no_appt.show(truncate=False)


+----------+------------+---------+---+------+-----------+----------------+--------------+-----------+----------+----------------+----------------+------+
|patient_id|patient_name|city     |age|gender|blood_group|insurance_status|appointment_id|doctor_name|department|appointment_date|consultation_fee|status|
+----------+------------+---------+---+------+-----------+----------------+--------------+-----------+----------+----------------+----------------+------+
|106       |Neha Singh  |NULL     |38 |Female|A+         |Inactive        |NULL          |NULL       |NULL      |NULL            |NULL            |NULL  |
|109       |Kiran Rao   |Hyderabad|33 |Male  |NULL       |Inactive        |NULL          |NULL       |NULL      |NULL            |NULL            |NULL  |
+----------+------------+---------+---+------+-----------+----------------+--------------+-----------+----------+----------------+----------------+------+



In [59]:
# 56
no_patient = appointments_df.join(
    patients_df,
    on='patient_id',
    how='left'
).filter(col('patient_name').isNull())
no_patient.show(truncate=False)


+----------+--------------+-----------+----------+----------------+----------------+---------+------------+----+----+------+-----------+----------------+
|patient_id|appointment_id|doctor_name|department|appointment_date|consultation_fee|status   |patient_name|city|age |gender|blood_group|insurance_status|
+----------+--------------+-----------+----------+----------------+----------------+---------+------------+----+----+------+-----------+----------------+
|120       |5009          |Dr. Ramesh |Cardiology|2025-02-05      |1500            |Completed|NULL        |NULL|NULL|NULL  |NULL       |NULL            |
+----------+--------------+-----------+----------+----------------+----------------+---------+------------+----+----+------+-----------+----------------+



In [60]:
# 57
appt_count = appointments_df.groupBy('patient_id').count()

appt_count.join(
    patients_df.select('patient_id', 'patient_name'),
    on='patient_id'
).select('patient_name', 'count').show()


+------------+-----+
|patient_name|count|
+------------+-----+
|Rahul Sharma|    2|
| Priya Reddy|    1|
| Sneha Patel|    1|
| Arjun Verma|    1|
| Nisha Reddy|    1|
|  Meera Nair|    1|
|  Amit Kumar|    1|
|  Farhan Ali|    1|
+------------+-----+



In [61]:
# 58
total_fees = appointments_df.groupBy('patient_id').sum('consultation_fee')
total_fees.join(
    patients_df.select('patient_id', 'patient_name'),
    on='patient_id'
).select(
    'patient_name',
    col('sum(consultation_fee)').alias('total_fees')
).show()


+------------+----------+
|patient_name|total_fees|
+------------+----------+
|Rahul Sharma|      2500|
| Priya Reddy|      2000|
| Sneha Patel|      2500|
| Arjun Verma|      2000|
| Nisha Reddy|      2500|
|  Meera Nair|      NULL|
|  Amit Kumar|      1500|
|  Farhan Ali|      1000|
+------------+----------+



In [62]:
# 59
total_fees = appointments_df.groupBy('patient_id').sum('consultation_fee')
total_fees.join(
    patients_df.select('patient_id', 'patient_name'),
    on='patient_id'
).select(
    'patient_name',
    col('sum(consultation_fee)').alias('total_fees')
).orderBy(col('total_fees').desc()).show(1)


+------------+----------+
|patient_name|total_fees|
+------------+----------+
| Sneha Patel|      2500|
+------------+----------+
only showing top 1 row


In [63]:
# 60
appt_count = appointments_df.groupBy('patient_id').count()

appt_count.join(
    patients_df.select('patient_id', 'patient_name'),
    on='patient_id'
).select(
    'patient_name',
    col('count').alias('appointment_count')
).show()


+------------+-----------------+
|patient_name|appointment_count|
+------------+-----------------+
|Rahul Sharma|                2|
| Priya Reddy|                1|
| Sneha Patel|                1|
| Arjun Verma|                1|
| Nisha Reddy|                1|
|  Meera Nair|                1|
|  Amit Kumar|                1|
|  Farhan Ali|                1|
+------------+-----------------+



In [64]:
# 61
fees_df = appointments_df.fillna({'consultation_fee': 0}).groupBy('patient_id').sum('consultation_fee').join(patients_df.select('patient_id', 'patient_name'), on='patient_id').select('patient_name', col('sum(consultation_fee)').alias('total_fees'))

w = Window.orderBy(col('total_fees').desc())
fees_df.withColumn('rank', rank().over(w)).show()


+------------+----------+----+
|patient_name|total_fees|rank|
+------------+----------+----+
|Rahul Sharma|      2500|   1|
| Sneha Patel|      2500|   1|
| Nisha Reddy|      2500|   1|
| Priya Reddy|      2000|   4|
| Arjun Verma|      2000|   4|
|  Amit Kumar|      1500|   6|
|  Farhan Ali|      1000|   7|
|  Meera Nair|         0|   8|
+------------+----------+----+



In [65]:
# 62
fees_df = appointments_df.fillna({'consultation_fee': 0}).groupBy('patient_id').sum('consultation_fee').join(patients_df.select('patient_id', 'patient_name'), on='patient_id').select('patient_name', col('sum(consultation_fee)').alias('total_fees'))

w = Window.orderBy(col('total_fees').desc())
fees_df.withColumn('dense_rank', dense_rank().over(w)).show()


+------------+----------+----------+
|patient_name|total_fees|dense_rank|
+------------+----------+----------+
|Rahul Sharma|      2500|         1|
| Sneha Patel|      2500|         1|
| Nisha Reddy|      2500|         1|
| Priya Reddy|      2000|         2|
| Arjun Verma|      2000|         2|
|  Amit Kumar|      1500|         3|
|  Farhan Ali|      1000|         4|
|  Meera Nair|         0|         5|
+------------+----------+----------+



In [66]:
# 63
fees_df = appointments_df.fillna({'consultation_fee': 0}).groupBy('patient_id').sum('consultation_fee').join(patients_df.select('patient_id', 'patient_name'), on='patient_id').select('patient_name', col('sum(consultation_fee)').alias('total_fees'))

w = Window.orderBy(col('total_fees').desc())
fees_df.withColumn('row_num', row_number().over(w)).show()


+------------+----------+-------+
|patient_name|total_fees|row_num|
+------------+----------+-------+
|Rahul Sharma|      2500|      1|
| Sneha Patel|      2500|      2|
| Nisha Reddy|      2500|      3|
| Priya Reddy|      2000|      4|
| Arjun Verma|      2000|      5|
|  Amit Kumar|      1500|      6|
|  Farhan Ali|      1000|      7|
|  Meera Nair|         0|      8|
+------------+----------+-------+



In [67]:
# 64
fees_df = appointments_df.fillna({'consultation_fee': 0}).groupBy('patient_id').sum('consultation_fee').join(patients_df.select('patient_id', 'patient_name'), on='patient_id').select('patient_name', col('sum(consultation_fee)').alias('total_fees'))

w = Window.orderBy(col('total_fees').desc())
fees_df.withColumn('rank', rank().over(w)).filter(col('rank') == 1).show()


+------------+----------+----+
|patient_name|total_fees|rank|
+------------+----------+----+
|Rahul Sharma|      2500|   1|
| Sneha Patel|      2500|   1|
| Nisha Reddy|      2500|   1|
+------------+----------+----+



In [ ]:
# 65
fees_df = appointments_df.fillna({'consultation_fee': 0}).groupBy('patient_id').sum('consultation_fee').join(patients_df.select('patient_id', 'patient_name'), on='patient_id').select('patient_name', col('sum(consultation_fee)').alias('total_fees'))

w = Window.orderBy(col('total_fees').desc())
fees_df.withColumn('rank', rank().over(w)).filter(col('rank') <= 3).show()


+------------+----------+----+
|patient_name|total_fees|rank|
+------------+----------+----+
|Rahul Sharma|      2500|   1|
| Sneha Patel|      2500|   1|
| Nisha Reddy|      2500|   1|
+------------+----------+----+



In [69]:
# 66
city_fees = appointments_df.fillna({'consultation_fee': 0}).join(patients_df.select('patient_id', 'patient_name', 'city'), on='patient_id').groupBy('patient_name', 'city').sum('consultation_fee').withColumnRenamed('sum(consultation_fee)', 'total_fees')

w = Window.partitionBy('city').orderBy(col('total_fees').desc())
city_fees.withColumn('rank', rank().over(w)).filter(col('rank') == 1).show()


+------------+---------+----------+----+
|patient_name|     city|total_fees|rank|
+------------+---------+----------+----+
| Nisha Reddy|Bangalore|      2500|   1|
| Sneha Patel|  Chennai|      2500|   1|
|  Farhan Ali|    Delhi|      1000|   1|
|Rahul Sharma|Hyderabad|      2500|   1|
|  Meera Nair|    Kochi|         0|   1|
|  Amit Kumar|   Mumbai|      1500|   1|
| Arjun Verma|     Pune|      2000|   1|
+------------+---------+----------+----+



In [70]:
# 67
city_fees = appointments_df.fillna({'consultation_fee': 0}).join(patients_df.select('patient_id', 'patient_name', 'city'), on='patient_id').groupBy('patient_name', 'city').sum('consultation_fee').withColumnRenamed('sum(consultation_fee)', 'total_fees')

w = Window.partitionBy('city').orderBy(col('total_fees').asc())
city_fees.withColumn('rank', rank().over(w)).filter(col('rank') == 1).show()


+------------+---------+----------+----+
|patient_name|     city|total_fees|rank|
+------------+---------+----------+----+
| Priya Reddy|Bangalore|      2000|   1|
| Sneha Patel|  Chennai|      2500|   1|
|  Farhan Ali|    Delhi|      1000|   1|
|Rahul Sharma|Hyderabad|      2500|   1|
|  Meera Nair|    Kochi|         0|   1|
|  Amit Kumar|   Mumbai|      1500|   1|
| Arjun Verma|     Pune|      2000|   1|
+------------+---------+----------+----+



In [71]:
# 68
w = Window.orderBy('appointment_id').rowsBetween(Window.unboundedPreceding, Window.currentRow)

appointments_df.fillna({'consultation_fee': 0}).withColumn('running_total', sum('consultation_fee').over(w)).show()


+--------------+----------+-----------+-----------+----------------+----------------+---------+-------------+
|appointment_id|patient_id|doctor_name| department|appointment_date|consultation_fee|   status|running_total|
+--------------+----------+-----------+-----------+----------------+----------------+---------+-------------+
|          5001|       101| Dr. Ramesh| Cardiology|      2025-01-10|            1500|Completed|         1500|
|          5002|       102| Dr. Suresh|  Neurology|      2025-01-11|            2000|Completed|         3500|
|          5003|       101|  Dr. Anita|Dermatology|      2025-01-15|            1000|Completed|         4500|
|          5004|       103| Dr. Ramesh| Cardiology|      2025-01-20|            1500|Cancelled|         6000|
|          5005|       104|  Dr. Priya|Orthopedics|      2025-01-22|            2500|Completed|         8500|
|          5006|       105|  Dr. Anita|Dermatology|      2025-01-25|            1000|  Pending|         9500|
|         

In [72]:
# 69
w = Window.orderBy('appointment_id')

appointments_df.fillna({'consultation_fee': 0}).withColumn('next_fee', lead('consultation_fee', 1).over(w)).show()


+--------------+----------+-----------+-----------+----------------+----------------+---------+--------+
|appointment_id|patient_id|doctor_name| department|appointment_date|consultation_fee|   status|next_fee|
+--------------+----------+-----------+-----------+----------------+----------------+---------+--------+
|          5001|       101| Dr. Ramesh| Cardiology|      2025-01-10|            1500|Completed|    2000|
|          5002|       102| Dr. Suresh|  Neurology|      2025-01-11|            2000|Completed|    1000|
|          5003|       101|  Dr. Anita|Dermatology|      2025-01-15|            1000|Completed|    1500|
|          5004|       103| Dr. Ramesh| Cardiology|      2025-01-20|            1500|Cancelled|    2500|
|          5005|       104|  Dr. Priya|Orthopedics|      2025-01-22|            2500|Completed|    1000|
|          5006|       105|  Dr. Anita|Dermatology|      2025-01-25|            1000|  Pending|    2000|
|          5007|       107| Dr. Suresh|  Neurology|    

In [73]:
# 70
w = Window.orderBy('appointment_id')

appointments_df.fillna({'consultation_fee': 0}).withColumn('prev_fee', lag('consultation_fee', 1).over(w)).show()


+--------------+----------+-----------+-----------+----------------+----------------+---------+--------+
|appointment_id|patient_id|doctor_name| department|appointment_date|consultation_fee|   status|prev_fee|
+--------------+----------+-----------+-----------+----------------+----------------+---------+--------+
|          5001|       101| Dr. Ramesh| Cardiology|      2025-01-10|            1500|Completed|    NULL|
|          5002|       102| Dr. Suresh|  Neurology|      2025-01-11|            2000|Completed|    1500|
|          5003|       101|  Dr. Anita|Dermatology|      2025-01-15|            1000|Completed|    2000|
|          5004|       103| Dr. Ramesh| Cardiology|      2025-01-20|            1500|Cancelled|    1000|
|          5005|       104|  Dr. Priya|Orthopedics|      2025-01-22|            2500|Completed|    1500|
|          5006|       105|  Dr. Anita|Dermatology|      2025-01-25|            1000|  Pending|    2500|
|          5007|       107| Dr. Suresh|  Neurology|    

In [74]:
# 71
prefs_df = spark.read.option('multiline', 'true').json(
    r'C:\new_env\hexaware-sql\june_11\patient_preferences.json'
)
prefs_df.show(truncate=False)


+-----------------------------+----------+------------------+
|contact                      |patient_id|preferred_hospital|
+-----------------------------+----------+------------------+
|{rahul@gmail.com, 9876500011}|101       |Apollo            |
|{priya@gmail.com, NULL}      |102       |Yashoda           |
|{NULL, 9876500013}           |103       |Care              |
|{sneha@gmail.com, 9876500014}|104       |NULL              |
+-----------------------------+----------+------------------+



In [75]:
# 72
prefs_df.printSchema()


root
 |-- contact: struct (nullable = true)
 |    |-- email: string (nullable = true)
 |    |-- phone: string (nullable = true)
 |-- patient_id: long (nullable = true)
 |-- preferred_hospital: string (nullable = true)



In [76]:
# 73
prefs_df.select('patient_id', col('contact.phone').alias('phone')).show()


+----------+----------+
|patient_id|     phone|
+----------+----------+
|       101|9876500011|
|       102|      NULL|
|       103|9876500013|
|       104|9876500014|
+----------+----------+



In [77]:
# 74
prefs_df.select('patient_id', col('contact.email').alias('email')).show()


+----------+---------------+
|patient_id|          email|
+----------+---------------+
|       101|rahul@gmail.com|
|       102|priya@gmail.com|
|       103|           NULL|
|       104|sneha@gmail.com|
+----------+---------------+



In [78]:
# 75
prefs_df.filter(col('contact.phone').isNull()).show(truncate=False)


+-----------------------+----------+------------------+
|contact                |patient_id|preferred_hospital|
+-----------------------+----------+------------------+
|{priya@gmail.com, NULL}|102       |Yashoda           |
+-----------------------+----------+------------------+



In [79]:
# 76
prefs_df.filter(col('contact.email').isNull()).show(truncate=False)


+------------------+----------+------------------+
|contact           |patient_id|preferred_hospital|
+------------------+----------+------------------+
|{NULL, 9876500013}|103       |Care              |
+------------------+----------+------------------+



In [80]:
# 77
prefs_df.filter(col('preferred_hospital').isNull()).show(truncate=False)


+-----------------------------+----------+------------------+
|contact                      |patient_id|preferred_hospital|
+-----------------------------+----------+------------------+
|{sneha@gmail.com, 9876500014}|104       |NULL              |
+-----------------------------+----------+------------------+



In [81]:
# 78
flat_prefs = prefs_df.select(
    'patient_id',
    'preferred_hospital',
    col('contact.phone').alias('phone'),
    col('contact.email').alias('email')
)
flat_prefs.fillna({'phone': 'Not Provided'}).show()


+----------+------------------+------------+---------------+
|patient_id|preferred_hospital|       phone|          email|
+----------+------------------+------------+---------------+
|       101|            Apollo|  9876500011|rahul@gmail.com|
|       102|           Yashoda|Not Provided|priya@gmail.com|
|       103|              Care|  9876500013|           NULL|
|       104|              NULL|  9876500014|sneha@gmail.com|
+----------+------------------+------------+---------------+



In [82]:
# 79
flat_prefs = prefs_df.select(
    'patient_id',
    'preferred_hospital',
    col('contact.phone').alias('phone'),
    col('contact.email').alias('email')
)
flat_prefs.fillna({'email': 'Not Provided'}).show()


+----------+------------------+----------+---------------+
|patient_id|preferred_hospital|     phone|          email|
+----------+------------------+----------+---------------+
|       101|            Apollo|9876500011|rahul@gmail.com|
|       102|           Yashoda|      NULL|priya@gmail.com|
|       103|              Care|9876500013|   Not Provided|
|       104|              NULL|9876500014|sneha@gmail.com|
+----------+------------------+----------+---------------+



In [83]:
# 80
flat_prefs = prefs_df.select(
    'patient_id',
    'preferred_hospital',
    col('contact.phone').alias('phone'),
    col('contact.email').alias('email')
)
flat_prefs.join(patients_df, on='patient_id', how='inner').show(truncate=False)


+----------+------------------+----------+---------------+------------+---------+---+------+-----------+----------------+
|patient_id|preferred_hospital|phone     |email          |patient_name|city     |age|gender|blood_group|insurance_status|
+----------+------------------+----------+---------------+------------+---------+---+------+-----------+----------------+
|101       |Apollo            |9876500011|rahul@gmail.com|Rahul Sharma|Hyderabad|35 |Male  |O+         |Active          |
|102       |Yashoda           |NULL      |priya@gmail.com|Priya Reddy |Bangalore|29 |Female|A+         |Active          |
|103       |Care              |9876500013|NULL           |Amit Kumar  |Mumbai   |42 |Male  |B+         |Inactive        |
|104       |NULL              |9876500014|sneha@gmail.com|Sneha Patel |Chennai  |31 |Female|O+         |Active          |
+----------+------------------+----------+---------------+------------+---------+---+------+-----------+----------------+



In [84]:
# 81
patients_df.createOrReplaceTempView('patients')


In [85]:
# 82
appointments_df.createOrReplaceTempView('appointments')


In [86]:
# 83
spark.sql('SELECT * FROM patients').show(truncate=False)


+----------+------------+---------+---+------+-----------+----------------+
|patient_id|patient_name|city     |age|gender|blood_group|insurance_status|
+----------+------------+---------+---+------+-----------+----------------+
|101       |Rahul Sharma|Hyderabad|35 |Male  |O+         |Active          |
|102       |Priya Reddy |Bangalore|29 |Female|A+         |Active          |
|103       |Amit Kumar  |Mumbai   |42 |Male  |B+         |Inactive        |
|104       |Sneha Patel |Chennai  |31 |Female|O+         |Active          |
|105       |Farhan Ali  |Delhi    |55 |Male  |AB+        |Active          |
|106       |Neha Singh  |NULL     |38 |Female|A+         |Inactive        |
|107       |Arjun Verma |Pune     |26 |Male  |B+         |Active          |
|108       |Meera Nair  |Kochi    |48 |Female|O-         |Active          |
|109       |Kiran Rao   |Hyderabad|33 |Male  |NULL       |Inactive        |
|110       |Nisha Reddy |Bangalore|41 |Female|A+         |Active          |
+----------+

In [87]:
# 84
spark.sql("SELECT * FROM patients WHERE city = 'Hyderabad'").show(truncate=False)


+----------+------------+---------+---+------+-----------+----------------+
|patient_id|patient_name|city     |age|gender|blood_group|insurance_status|
+----------+------------+---------+---+------+-----------+----------------+
|101       |Rahul Sharma|Hyderabad|35 |Male  |O+         |Active          |
|109       |Kiran Rao   |Hyderabad|33 |Male  |NULL       |Inactive        |
+----------+------------+---------+---+------+-----------+----------------+



In [88]:
# 85
spark.sql('SELECT city, COUNT(*) as count FROM patients GROUP BY city').show()


+---------+-----+
|     city|count|
+---------+-----+
|Hyderabad|    2|
|Bangalore|    2|
|     Pune|    1|
|     NULL|    1|
|   Mumbai|    1|
|  Chennai|    1|
|    Delhi|    1|
|    Kochi|    1|
+---------+-----+



In [89]:
# 86
spark.sql('SELECT department, COUNT(*) as count FROM appointments GROUP BY department').show()


+-----------+-----+
| department|count|
+-----------+-----+
| Cardiology|    3|
|  Neurology|    2|
|Dermatology|    3|
|Orthopedics|    2|
+-----------+-----+



In [90]:
# 87
spark.sql('SELECT department, AVG(consultation_fee) as avg_fee FROM appointments GROUP BY department').show()


+-----------+-------+
| department|avg_fee|
+-----------+-------+
| Cardiology| 1500.0|
|  Neurology| 2000.0|
|Dermatology| 1000.0|
|Orthopedics| 2500.0|
+-----------+-------+



In [91]:
# 88
spark.sql('SELECT MAX(consultation_fee) as highest_fee FROM appointments').show()


+-----------+
|highest_fee|
+-----------+
|       2500|
+-----------+



In [92]:
# 89
spark.sql('SELECT patient_id, COUNT(*) as appt_count FROM appointments GROUP BY patient_id').show()


+----------+----------+
|patient_id|appt_count|
+----------+----------+
|       101|         2|
|       102|         1|
|       104|         1|
|       107|         1|
|       110|         1|
|       108|         1|
|       103|         1|
|       105|         1|
|       120|         1|
+----------+----------+



In [93]:
# 90
spark.sql('''
    SELECT p.patient_name, SUM(a.consultation_fee) as total_spent
    FROM appointments a
    JOIN patients p ON a.patient_id = p.patient_id
    GROUP BY p.patient_name
    ORDER BY total_spent DESC
    LIMIT 5
''').show()


+------------+-----------+
|patient_name|total_spent|
+------------+-----------+
|Rahul Sharma|       2500|
| Sneha Patel|       2500|
| Nisha Reddy|       2500|
| Arjun Verma|       2000|
| Priya Reddy|       2000|
+------------+-----------+



In [94]:
# 91
patients_df = spark.read.csv(r'C:\new_env\hexaware-sql\june_11\patients.csv', header=True, inferSchema=True)
appointments_df = spark.read.csv(r'C:\new_env\hexaware-sql\june_11\appointments.csv', header=True, inferSchema=True)
print("Files read successfully")


Files read successfully


In [95]:
# 92
prefs_df = spark.read.option('multiline', 'true').json(r'C:\new_env\hexaware-sql\june_11\patient_preferences.json')
print("JSON read successfully")


JSON read successfully


In [96]:
# 93
patients_clean = patients_df.fillna({
    'city': 'Unknown',
    'blood_group': 'Not Available'
})
appointments_clean = appointments_df.fillna({'consultation_fee': 0})
print("Nulls handled")


Nulls handled


In [97]:
# 94
joined_df = patients_clean.join(
    appointments_clean,
    on='patient_id',
    how='inner'
)
joined_df.show(truncate=False)


+----------+------------+---------+---+------+-----------+----------------+--------------+-----------+-----------+----------------+----------------+---------+
|patient_id|patient_name|city     |age|gender|blood_group|insurance_status|appointment_id|doctor_name|department |appointment_date|consultation_fee|status   |
+----------+------------+---------+---+------+-----------+----------------+--------------+-----------+-----------+----------------+----------------+---------+
|101       |Rahul Sharma|Hyderabad|35 |Male  |O+         |Active          |5001          |Dr. Ramesh |Cardiology |2025-01-10      |1500            |Completed|
|102       |Priya Reddy |Bangalore|29 |Female|A+         |Active          |5002          |Dr. Suresh |Neurology  |2025-01-11      |2000            |Completed|
|101       |Rahul Sharma|Hyderabad|35 |Male  |O+         |Active          |5003          |Dr. Anita  |Dermatology|2025-01-15      |1000            |Completed|
|103       |Amit Kumar  |Mumbai   |42 |Male  |

In [98]:
# 95
final_df = joined_df.withColumn(
    'age_group',
    when(col('age') < 18, 'Minor')
    .when(col('age') < 40, 'Adult')
    .when(col('age') < 60, 'Middle-Aged')
    .otherwise('Senior')
)
final_df.select('patient_name', 'age', 'age_group').show()


+------------+---+-----------+
|patient_name|age|  age_group|
+------------+---+-----------+
|Rahul Sharma| 35|      Adult|
|Rahul Sharma| 35|      Adult|
| Priya Reddy| 29|      Adult|
|  Amit Kumar| 42|Middle-Aged|
| Sneha Patel| 31|      Adult|
|  Farhan Ali| 55|Middle-Aged|
| Arjun Verma| 26|      Adult|
|  Meera Nair| 48|Middle-Aged|
| Nisha Reddy| 41|Middle-Aged|
+------------+---+-----------+



In [99]:
# 96
revenue_df = appointments_clean.groupBy('department').agg(
    sum('consultation_fee').alias('total_revenue'),
    avg('consultation_fee').alias('avg_revenue'),
    count('appointment_id').alias('total_appointments')
)
revenue_df.show()


+-----------+-------------+-----------------+------------------+
| department|total_revenue|      avg_revenue|total_appointments|
+-----------+-------------+-----------------+------------------+
| Cardiology|         4500|           1500.0|                 3|
|  Neurology|         4000|           2000.0|                 2|
|Dermatology|         2000|666.6666666666666|                 3|
|Orthopedics|         5000|           2500.0|                 2|
+-----------+-------------+-----------------+------------------+



In [ ]:
# 97
patient_spending = appointments_clean.groupBy('patient_id').sum('consultation_fee').join(patients_clean.select('patient_id', 'patient_name'), on='patient_id').select(
        'patient_name',
        col('sum(consultation_fee)').alias('total_spent')
    ).orderBy(col('total_spent').desc())

patient_spending.show()


+------------+-----------+
|patient_name|total_spent|
+------------+-----------+
|Rahul Sharma|       2500|
| Sneha Patel|       2500|
| Nisha Reddy|       2500|
| Priya Reddy|       2000|
| Arjun Verma|       2000|
|  Amit Kumar|       1500|
|  Farhan Ali|       1000|
|  Meera Nair|          0|
+------------+-----------+



In [ ]:
# 98
dept_revenue = appointments_clean.groupBy('department').sum('consultation_fee').withColumnRenamed('sum(consultation_fee)', 'dept_revenue').orderBy(col('dept_revenue').desc())

dept_revenue.show()


+-----------+------------+
| department|dept_revenue|
+-----------+------------+
|Orthopedics|        5000|
| Cardiology|        4500|
|  Neurology|        4000|
|Dermatology|        2000|
+-----------+------------+



In [102]:
# 99
final_df.write.mode('overwrite').parquet(
    r'C:\new_env\hexaware-sql\june_11\hospital_analytics_final.parquet'
)
print("Saved as hospital_analytics_final.parquet")


Saved as hospital_analytics_final.parquet


In [ ]:
# 100
total_patients = patients_df.count()
total_appointments = appointments_df.count()
total_revenue = appointments_clean.agg(sum('consultation_fee')).collect()[0][0]
avg_fee = appointments_clean.agg(avg('consultation_fee')).collect()[0][0]

top_dept = appointments_clean.groupBy('department').sum('consultation_fee').orderBy(col('sum(consultation_fee)').desc()).first()[0]

top_patient = appointments_clean.groupBy('patient_id').sum('consultation_fee').join(patients_clean.select('patient_id', 'patient_name'), on='patient_id').orderBy(col('sum(consultation_fee)').desc()).first()[1]

print("=" * 50)
print("     HOSPITAL ANALYTICS REPORT")
print("=" * 50)
print(f"Total Patients          : {total_patients}")
print(f"Total Appointments      : {total_appointments}")
print(f"Total Revenue           : {total_revenue}")
print(f"Average Fee             : {round(avg_fee, 2)}")
print(f"Top Department          : {top_dept}")
print(f"Highest Spending Patient: {top_patient}")
print("=" * 50)


     HOSPITAL ANALYTICS REPORT
Total Patients          : 10
Total Appointments      : 10
Total Revenue           : 15500
Average Fee             : 1550.0
Top Department          : Orthopedics
Highest Spending Patient: 2500
